# 实验四 · 哲学家就餐：死锁、活锁与资源分级

**所属**：《并行计算》第四章 · Pthread 多线程编程　|　**难度**：⭐⭐⭐ 进阶　|　**预计时长**：20–30 分钟

> **实验说明**
> 1. 实验三中，每个线程在任一时刻只需要持有**一把**锁。本实验把条件改为：每个线程必须**同时持有两把**锁才能工作。仅此一个变化，就足以引出并发编程中最著名的两类故障——**死锁**与**活锁**。
> 2. 实验流程为：用最自然的写法制造一次真实的死锁 → 破坏「占有并等待」条件，却掉入活锁 → 破坏「循环等待」条件，得到正确解法。三种策略层层递进。
> 3. **本实验的前两种策略不会自行终止**。Notebook 中的所有运行都设有超时保护，程序会被自动终止，这是预期行为，不是故障。
> 4. 请自上而下依次执行各单元格（Shift+Enter）。
> 5. 遇到 🔧 **动手练习** 与 🤔 **思考题** 时，建议先独立完成，再阅读后续内容。

## 🎯 学习目标

完成本实验后，学生应能够：

- 复述死锁的**四个必要条件**（Coffman 条件），并在代码中逐一对应
- 严格区分**死锁**与**活锁**：前者线程全部阻塞，后者线程都在运行却没有实质进展
- 使用 `pthread_mutex_trylock` 破坏「占有并等待」，并说明它为何只能消除死锁、却不能保证所有线程最终完成
- 运用**资源分级**策略破坏「循环等待」，得到有保证的解法
- 把「为所有锁定义全局统一的获取顺序」总结为可执行的代码规范
- 掌握诊断挂起程序的基本方法：输出时间序列、CPU 占用与线程栈

## 🗺️ 学习路径

1. **准备阶段**：理解哲学家就餐模型，明确它抽象的是「一个操作需要同时持有多个资源」这一普遍情形
2. **理论分析**：掌握 Coffman 四条件，认识到破坏任意一个即可预防死锁
3. **策略零（死锁）**：先左后右、阻塞等待，观察程序完全挂起
   → 学习如何判定一次挂起确实是死锁
4. **策略一（活锁）**：改用 `trylock` 失败即释放，死锁消除但可能陷入活锁
   → 建立「不死锁」与「一定能完成」是两回事的认识
5. **策略二（解法）**：按叉子编号从小到大获取，循环等待在数学上被排除
   → 导出全局加锁顺序这一工程规范，该规范将在实验六、实验八再次出现

## 1. 背景与动机

实验三用互斥量解决了数据竞争：多个线程累加到同一个变量时，用一把锁保护临界区即可。在那个场景中，**每个线程在任一时刻只需要持有一把锁**。

本实验把条件改为：**每个线程必须同时持有两把锁才能完成工作**。这一个变化带来的不是量的增加，而是质的改变——即使每一把锁的使用都完全正确，程序整体仍可能永久停滞。

这并非人为构造的特例。真实系统中大量操作都需要同时持有多个资源：

<!--
| 场景 | 需要同时持有的资源 |
|---|---|
| 银行转账 | 转出账户与转入账户两把锁 |
| 数据库多表事务 | 涉及的每张表（或每行）的锁 |
| 文件系统重命名 | 源目录与目标目录的 inode 锁 |
| 图形界面重绘 | 数据模型锁与视图锁 |
-->
<table>
  <thead>
    <tr>
      <th style="text-align: left;">场景</th>
      <th style="text-align: left;">需要同时持有的资源</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;">银行转账</td>
      <td style="text-align: left;">转出账户与转入账户两把锁</td>
    </tr>
    <tr>
      <td style="text-align: left;">数据库多表事务</td>
      <td style="text-align: left;">涉及的每张表（或每行）的锁</td>
    </tr>
    <tr>
      <td style="text-align: left;">文件系统重命名</td>
      <td style="text-align: left;">源目录与目标目录的 inode 锁</td>
    </tr>
    <tr>
      <td style="text-align: left;">图形界面重绘</td>
      <td style="text-align: left;">数据模型锁与视图锁</td>
    </tr>
  </tbody>
</table>

因此，掌握多锁场景下的死锁预防，是并发编程从「会用锁」走向「用对锁」的必经一步。

> 实验三说明了互斥量**能**解决什么；本实验说明它**用不好会**造成什么。
> 两者合起来，才构成对互斥量这一工具的完整认识。

## 2. 问题模型：哲学家就餐

Dijkstra 于 1965 年提出的经典模型：五位哲学家围坐圆桌，相邻两人之间只有一支叉子，共五支。哲学家交替地思考与进餐，**进餐必须同时取得左右两支叉子**。

```
                        叉子 0
                 P0 ──────────── P1
                ╱                  ╲
        叉子 4 ╱                    ╲ 叉子 1
              ╱                      ╲
            P4                        P2
              ╲                      ╱
        叉子 3 ╲                    ╱ 叉子 2
                ╲                  ╱
                 ╲──────  P3 ─────╱

        P = 哲学家（线程）    叉子 = 互斥量
```

哲学家 $i$ 需要**叉子 $i$**（左）与**叉子 $(i+1) \bmod 5$**（右）。相邻两位哲学家争抢他们之间的那一支叉子。

代码中每支叉子就是一个互斥量：

```c
#define NUM_PHILOSOPHERS 5
pthread_mutex_t forks[NUM_PHILOSOPHERS];
```

### 本实验的三种取叉策略

<!--
| 策略 | 做法 | 结果 |
|---|---|---|
| **策略零** | 先取左叉，阻塞等待右叉 | **死锁** |
| **策略一** | 取左叉后 `trylock` 右叉，失败则放下左叉重试 | **活锁**（概率性） |
| **策略二** | 总是先取编号较小的叉子 | **正确** |
-->
<table>
  <thead>
    <tr>
      <th style="text-align: left;">策略</th>
      <th style="text-align: left;">做法</th>
      <th style="text-align: left;">结果</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;"><strong>策略零</strong></td>
      <td style="text-align: left;">先取左叉，阻塞等待右叉</td>
      <td style="text-align: left;"><strong>死锁</strong></td>
    </tr>
    <tr>
      <td style="text-align: left;"><strong>策略一</strong></td>
      <td style="text-align: left;">取左叉后 <code>trylock</code> 右叉，失败则放下左叉重试</td>
      <td style="text-align: left;"><strong>活锁</strong>（概率性）</td>
    </tr>
    <tr>
      <td style="text-align: left;"><strong>策略二</strong></td>
      <td style="text-align: left;">总是先取编号较小的叉子</td>
      <td style="text-align: left;"><strong>正确</strong></td>
    </tr>
  </tbody>
</table>

## 3. 死锁的四个必要条件

死锁的成立需要以下四个条件**同时**满足，这一结论由 Coffman 等人于 1971 年给出：

<!--
| 条件 | 含义 | 在本模型中的体现 |
|---|---|---|
| **互斥**（Mutual Exclusion） | 资源同一时刻只能被一个线程占有 | 一支叉子同时只能由一人持有 |
| **占有并等待**（Hold and Wait） | 已持有资源的线程仍在等待其他资源 | 拿着左叉等待右叉 |
| **不可抢占**（No Preemption） | 资源只能由持有者主动释放 | 不能夺取他人手中的叉子 |
| **循环等待**（Circular Wait） | 存在一条首尾相接的等待链 | $P_0 \to P_1 \to P_2 \to P_3 \to P_4 \to P_0$ |
-->
<table>
  <thead>
    <tr>
      <th style="text-align: left;">条件</th>
      <th style="text-align: left;">含义</th>
      <th style="text-align: left;">在本模型中的体现</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;"><strong>互斥</strong>（Mutual Exclusion）</td>
      <td style="text-align: left;">资源同一时刻只能被一个线程占有</td>
      <td style="text-align: left;">一支叉子同时只能由一人持有</td>
    </tr>
    <tr>
      <td style="text-align: left;"><strong>占有并等待</strong>（Hold and Wait）</td>
      <td style="text-align: left;">已持有资源的线程仍在等待其他资源</td>
      <td style="text-align: left;">拿着左叉等待右叉</td>
    </tr>
    <tr>
      <td style="text-align: left;"><strong>不可抢占</strong>（No Preemption）</td>
      <td style="text-align: left;">资源只能由持有者主动释放</td>
      <td style="text-align: left;">不能夺取他人手中的叉子</td>
    </tr>
    <tr>
      <td style="text-align: left;"><strong>循环等待</strong>（Circular Wait）</td>
      <td style="text-align: left;">存在一条首尾相接的等待链</td>
      <td style="text-align: left;">P₀ → P₁ → P₂ → P₃ → P₄ → P₀</td>
    </tr>
  </tbody>
</table>


**四个条件必须同时成立，死锁才会发生；因此破坏其中任意一个，即可预防死锁。**

其中「互斥」与「不可抢占」是互斥量的固有语义，无法改变——若资源可以共享，就不需要锁；若可以强行剥夺，锁也就失去了意义。因此实际可以着手的只有后两个条件：

- **策略一**破坏「占有并等待」：拿不到右叉就主动放下左叉；
- **策略二**破坏「循环等待」：规定统一的获取顺序，使等待链无法闭合。

本实验将依次验证这两条路径，并说明为何只有后者是可靠的。

## 4. 环境准备

本实验不做性能测量，因此无需核心同构性检查。但**核心数会影响死锁与活锁的显现概率**：核心数越多，线程真正并发执行的程度越高，两种故障越容易复现。

In [ ]:
import platform, subprocess, shutil, sys, os, re, time

print("Python  :", sys.version.split()[0])
print("架构    :", platform.machine())
CC = shutil.which("gcc") or shutil.which("clang") or shutil.which("cc")
print("编译器  :", CC)
NCPU = os.cpu_count()
print("CPU 核心:", NCPU)
print("stdbuf  :", shutil.which("stdbuf") or "未找到（见下方说明）")

if CC is None:
    print(
        "\n⚠️  未找到 C 编译器，请先安装 gcc（如 sudo apt install build-essential）。"
    )
elif NCPU == 1:
    print("\n⚠️  当前仅 1 个核心：线程只能分时轮转，活锁很可能不显现。")
    print("    死锁通常仍可复现。建议在华为鲲鹏多核处理器上完整重做本实验。")
else:
    print(f"\n✅ 环境就绪：编译器可用，{NCPU} 核可用，可以开始实验！")


### ⚠️ 一个必须处理的实际问题：输出缓冲

本实验的前两种策略无法自行结束，运行时只能设定超时并强制终止进程。这带来一个容易被忽视的陷阱：

C 标准库对 `stdout` 采用**行缓冲**还是**全缓冲**，取决于它是否连接到终端。在终端下运行时是行缓冲，每行立即输出；而 Notebook 通过**管道**捕获输出，此时 `stdout` 变为**全缓冲**——数据先积存在用户态缓冲区中，直到缓冲区满或程序正常退出才写出。

后果是：被强制终止的进程，其缓冲区内容**全部丢失**，我们将看不到任何输出。

解决办法是用 `stdbuf -oL` 强制行缓冲：

```bash
stdbuf -oL ./dining 0        # -oL 表示 stdout 使用行缓冲
```

本实验的运行函数已内置该处理。若系统中没有 `stdbuf`，也可在 C 代码中于 `main` 开头调用 `setvbuf(stdout, NULL, _IOLBF, 0)` 达到同样效果。

### 编译与监测工具函数

除常规的编译与运行外，本实验还需要一个**带超时的监测函数**。它在程序运行期间周期性采样三项指标：

- **累计输出行数**——反映程序是否还在产生输出；
- **完成进餐次数**——反映程序是否取得**实质进展**；
- **累计 CPU 时间**——反映线程是在阻塞还是在执行。

这三项指标合起来，足以区分死锁、活锁与正常完成。

In [2]:
SRC_DIR = "src_philosophers"
os.makedirs(SRC_DIR, exist_ok=True)
TICK = os.sysconf("SC_CLK_TCK")


def compile_c(src, out):
    """用全章统一选项编译一个源文件，成功返回可执行文件名，失败返回 None。"""
    base = shutil.which("gcc") or shutil.which("cc") or "cc"
    cmd = f"{base} -O3 -fPIC -pthread -Wall -Wextra {src} -o {out} -lpthread -lm"
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode == 0:
        print("✅ 编译成功：", cmd)
        if r.stderr.strip():
            print(r.stderr.strip())
        return out
    print("❌ 编译失败：\n", r.stderr)
    return None


def _cpu_seconds(pid):
    """读取 /proc/<pid>/stat，返回该进程全部线程累计的 CPU 时间（秒）。"""
    try:
        fields = open(f"/proc/{pid}/stat").read().rsplit(")", 1)[1].split()
        return (int(fields[11]) + int(fields[12])) / TICK  # utime + stime
    except (OSError, IndexError, ValueError):
        return 0.0


def monitor(binary, *args, timeout=8.0, interval=1.0):
    """带超时地运行程序并周期性采样，返回 (是否挂起, 采样序列, 完整输出)。

    采样项为 (时刻秒, 累计输出行数, 完成进餐次数, 累计CPU秒)。
    使用 stdbuf -oL 强制行缓冲，确保被终止时已产生的输出不会丢失。
    """
    log_path = os.path.join(SRC_DIR, "run.log")
    prefix = ["stdbuf", "-oL"] if shutil.which("stdbuf") else []
    with open(log_path, "w") as log:
        p = subprocess.Popen(
            prefix + ["./" + binary] + [str(a) for a in args],
            stdout=log,
            stderr=subprocess.STDOUT,
        )
        samples, t0 = [], time.time()
        while time.time() - t0 < timeout and p.poll() is None:
            time.sleep(interval)
            text = open(log_path, errors="replace").read()
            samples.append(
                (
                    round(time.time() - t0, 1),
                    len(text.splitlines()),
                    text.count("finished a meal"),
                    round(_cpu_seconds(p.pid), 2),
                )
            )
        hung = p.poll() is None
        if hung:
            p.kill()
        p.wait()
    return hung, samples, open(log_path, errors="replace").read()


def report(title, hung, samples, text, expect_meals=15):
    """打印一次监测的结论。"""
    lines = len(text.splitlines())
    meals = text.count("finished a meal")
    print(f"═══ {title} ═══")
    print(f"{'时刻(s)':>8}{'输出行数':>10}{'完成进餐':>10}{'CPU(s)':>10}")
    for t, n, m, c in samples:
        print(f"{t:>8.1f}{n:>10}{m:>10}{c:>10.2f}")
    print(
        f"\n结果：{'超时被强制终止（程序未能自行结束）' if hung else '程序自行正常结束'}"
    )
    print(f"总输出 {lines} 行，完成进餐 {meals} / {expect_meals} 次")
    return hung, lines, meals


## 5. 三种策略的设计

本实验的程序把三种策略编译在同一个可执行文件中，由命令行参数选择：

```bash
./dining 0     # 策略零：先左后右，阻塞等待
./dining 1     # 策略一：trylock 失败即重试
./dining 2     # 策略二：资源分级
```

三种策略共用同一套哲学家主循环与相同的思考、进餐时长，因此观察到的差异只可能来自取叉方式本身。本节先分别剖析三者的设计意图，第 6 节再给出完整源码。

### 5.1 策略零：先左后右，阻塞等待

最自然的写法：

```c
pthread_mutex_lock(&forks[left]);       // 先取左叉
sleep_ms(100);                          // 制造一个上下文切换窗口
pthread_mutex_lock(&forks[right]);      // 再取右叉，阻塞等待
/* 进餐 */
pthread_mutex_unlock(&forks[right]);
pthread_mutex_unlock(&forks[left]);
```

若五位哲学家几乎同时取起各自的左叉，五支叉子便全部被占用，每个人都在等待右邻居手中的那一支——而右邻居也在等待**他的**右邻居。等待链首尾相接，**所有人永远等下去**。

对照第 3 节的四个条件：互斥（叉子独占）、占有并等待（持左叉等右叉）、不可抢占（不能夺取）、循环等待（$P_0 \to \cdots \to P_4 \to P_0$）——四者全部成立。

### 💡 关于那行 `sleep_ms(100)`

它是**刻意加入**的。没有这段延迟，某个抢先的哲学家可能在其他人开始之前就取得两支叉子并完成进餐，死锁便不一定复现。加入延迟后，五个线程几乎必然停留在「各持一支左叉」的状态，死锁得以稳定重现。

> **教学代码需要可复现的失败。** 真实系统中的死锁往往数月才出现一次，正因如此才格外难以排查。本实验用一行 `sleep` 把概率事件变为必然事件，便于课堂观察。

### 5.2 策略一：trylock 失败即重试

既然「占有并等待」是四条件之一，那就破坏它——取不到右叉时，把左叉**放回去**：

```c
while (!ate) {
  pthread_mutex_lock(&forks[left]);

  sleep_ms(100);                                   // 保证所有人都已持有左叉

  if (pthread_mutex_trylock(&forks[right]) == 0) { // 非阻塞地尝试
    /* 两支都到手，进餐 */
    ate = 1;
  } else {
    pthread_mutex_unlock(&forks[left]);            // 放回左叉，立即重试
  }
}
```

`pthread_mutex_trylock` 与 `pthread_mutex_lock` 的区别在于：**它永不阻塞**。锁可用时获取并返回 0，锁已被占用时立即返回 `EBUSY`。

```c
int pthread_mutex_trylock(pthread_mutex_t *mutex);   // 成功返回 0，忙则返回 EBUSY
```

**死锁确实被消除了**：线程不再抱着左叉无限期等待右叉，四个必要条件不再同时成立。

### ⚠️ 但引入了新的问题：活锁

若五位哲学家的节奏恰好一致，可能出现如下循环：

1. 所有人取起左叉；
2. 延迟期间，五支叉子全部处于被持有状态；
3. 所有人尝试取右叉，全部失败；
4. 所有人放下左叉，立即重新开始；
5. 回到第 1 步。

**没有任何线程被阻塞，所有线程都在正常运行，但没有一个人吃到饭。** 这种状态称为**活锁**（livelock）。

💡 此处的 sleep_ms(100) 与策略零同理

与策略零一样，这段延迟是刻意加入的。没有它，取左叉与试右叉之间的间隔只有几微秒，某位抢先的哲学家会在其他人反应过来之前就取得两支叉子并完成进餐，活锁便不会形成。加入延迟后，五位哲学家必然同时处于「各持一支左叉」的状态，随后的 trylock 也就必然全部失败，活锁得以稳定重现。

> 注意：策略零的延迟制造的是死锁，本策略的延迟制造的是活锁。二者延迟的作用相同——放大并发窗口，使故障可复现；差别只在于取右叉时用的是阻塞的 lock 还是非阻塞的 trylock。

本实现在失败后立即重试，没有任何退避等待。工程上常用的缓解手段是随机退避（randomized backoff）：失败后等待一段随机时长再重试，以打破线程之间的对称性。但请注意，随机退避只是降低概率，并不能从原理上保证每个线程最终都能取得进展。


### 5.3 策略二：资源分级

再次回到四个条件，这次破坏**循环等待**。

Dijkstra 提出的**资源分级**（resource hierarchy）思想极为简单：**给所有资源统一编号，规定任何线程都必须按编号从小到大依次申请。**

```c
int first  = (left < right) ? left  : right;    // 先取编号较小的
int second = (left < right) ? right : left;

pthread_mutex_lock(&forks[first]);
pthread_mutex_lock(&forks[second]);
```

### 为什么等待环一定会被打破

哲学家 $P_0 \sim P_3$ 的左叉编号都小于右叉，行为与策略零相同。关键在于**哲学家 $P_4$**：他的左叉是 4、右叉是 0，按新规则必须**先取叉子 0**。

于是等待链 $P_0 \to P_1 \to P_2 \to P_3 \to P_4 \to P_0$ 中的最后一环被切断：$P_4$ 不再等待 $P_0$ 手中的叉子，而是与 $P_0$ **争抢同一支叉子 0**。争抢必有胜负，**失败的一方不持有任何叉子**，因而不会阻塞他人。环无法闭合。

**一般性论证**：若所有线程都按编号递增的顺序获取锁，则任何一个「持有锁 $k$ 并等待锁 $m$」的线程必有 $m > k$。于是等待关系构成一个**严格递增的偏序**，而偏序关系不可能成环。因此死锁在数学上被排除，与调度时序无关。

> **工程规范：需要同时持有多把锁时，必须为所有锁定义全局统一的获取顺序，并在代码评审中强制执行。**
>
> 编号可以取锁的地址、对象 ID，或任何全局一致的标准。关键在于**全局一致**。

这条规范在本章反复出现：

<!--
| 场景 | 全局顺序的形式 |
|---|---|
| 本实验 | 先取编号较小的叉子 |
| 实验八 · 并发链表交接锁 | 按链表的物理先后顺序逐节点加锁 |
-->
<table>
  <thead>
    <tr>
      <th style="text-align: left;">场景</th>
      <th style="text-align: left;">全局顺序的形式</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;">本实验</td>
      <td style="text-align: left;">先取编号较小的叉子</td>
    </tr>
    <tr>
      <td style="text-align: left;">实验八 · 并发链表交接锁</td>
      <td style="text-align: left;">按链表的物理先后顺序逐节点加锁</td>
    </tr>
  </tbody>
</table>

三者是同一条规范在不同场景下的体现。

## 6. 完整源码与编译

下面用 `%%writefile` 将完整源码写入 `src_philosophers/pthread_dining_philosophers.c`。

In [ ]:
%%writefile {SRC_DIR}/pthread_dining_philosophers.c
#include <pthread.h>
#include <stdio.h>
#include <stdlib.h>
#include <unistd.h>

#define NUM_PHILOSOPHERS 5
#define MEALS_TO_EAT 3

// One mutex per fork. Philosopher i needs fork i (left) and fork (i+1)%5
// (right), so neighbours compete for the fork between them.
pthread_mutex_t forks[NUM_PHILOSOPHERS];
int strategy = 0;  // 0 = deadlock, 1 = livelock, 2 = resource hierarchy

static void sleep_ms(int milliseconds) { usleep(milliseconds * 1000); }

// Strategy 0: everyone takes the left fork first, then the right one.
// All four Coffman conditions hold, and the sleep between the two acquisitions
// makes the circular wait almost certain: every philosopher ends up holding
// one fork and waiting forever for a neighbour's.
static void eat_naive_deadlock(long id, int left, int right) {
  pthread_mutex_lock(&forks[left]);
  printf("[Phil %ld] picked up LEFT fork %d\n", id, left);

  // Without this delay a fast philosopher might grab both forks and finish
  // before the others start, hiding the deadlock.
  sleep_ms(100);

  printf("[Phil %ld] waiting for RIGHT fork %d ...\n", id, right);
  pthread_mutex_lock(&forks[right]);
  printf("[Phil %ld] picked up RIGHT fork %d\n", id, right);

  printf("[Phil %ld] eating\n", id);
  sleep_ms(100);

  pthread_mutex_unlock(&forks[right]);
  pthread_mutex_unlock(&forks[left]);
  printf("[Phil %ld] finished a meal\n", id);
}

// Strategy 1: take the left fork, hold it briefly, then try the right one; if
// it is busy, put the left one back and retry immediately. This breaks "hold
// and wait", so it cannot deadlock -- but if the philosophers stay in step they
// keep picking up and putting down forks forever. The threads are running; they
// just make no progress. That is livelock, not deadlock.
static void eat_polite_livelock(long id, int left, int right) {
  int ate = 0;
  while (!ate) {
    pthread_mutex_lock(&forks[left]);
    printf("[Phil %ld] picked up LEFT fork %d\n", id, left);

    // Same purpose as in strategy 0: it guarantees that every philosopher is
    // holding a left fork before anyone reaches for a right one, which is what
    // makes the livelock reproducible.    
    sleep_ms(100);

    if (pthread_mutex_trylock(&forks[right]) == 0) {
      printf("[Phil %ld] picked up RIGHT fork %d\n", id, right);
      printf("[Phil %ld] eating\n", id);
      sleep_ms(100);

      pthread_mutex_unlock(&forks[right]);
      pthread_mutex_unlock(&forks[left]);
      printf("[Phil %ld] finished a meal\n", id);
      ate = 1;
    } else {
      printf("[Phil %ld] RIGHT fork %d busy, releasing LEFT %d and retrying\n",
            id, right, left);
      pthread_mutex_unlock(&forks[left]);
    }
  }
}

// Strategy 2: resource hierarchy (Dijkstra). Number the forks and always take
// the lower-numbered one first. This breaks "circular wait": the last
// philosopher reaches for fork 0 first instead of fork 4, so the cycle in the
// wait-for graph cannot close. This is the same rule as "define one global
// lock acquisition order".
static void eat_hierarchy_solution(long id, int left, int right) {
  int first = (left < right) ? left : right;
  int second = (left < right) ? right : left;

  pthread_mutex_lock(&forks[first]);
  printf("[Phil %ld] picked up fork %d (lower id first)\n", id, first);

  pthread_mutex_lock(&forks[second]);
  printf("[Phil %ld] picked up fork %d\n", id, second);

  printf("[Phil %ld] eating\n", id);
  sleep_ms(100);

  pthread_mutex_unlock(&forks[second]);
  pthread_mutex_unlock(&forks[first]);
  printf("[Phil %ld] finished a meal\n", id);
}

void* Philosopher(void* rank) {
  long id = (long)rank;
  int left_fork = (int)id;
  int right_fork = (int)((id + 1) % NUM_PHILOSOPHERS);

  for (int i = 0; i < MEALS_TO_EAT; ++i) {
    printf("[Phil %ld] thinking\n", id);
    sleep_ms(50);

    switch (strategy) {
      case 0:
        eat_naive_deadlock(id, left_fork, right_fork);
        break;
      case 1:
        eat_polite_livelock(id, left_fork, right_fork);
        break;
      default:
        eat_hierarchy_solution(id, left_fork, right_fork);
        break;
    }
  }
  printf("[Phil %ld] full, leaving the table\n", id);
  return NULL;
}

int main(int argc, char* argv[]) {
  if (argc != 2) {
    fprintf(stderr, "Usage: %s <strategy>\n", argv[0]);
    fprintf(stderr, "  0: deadlock  (left fork first, then right)\n");
    fprintf(stderr, "  1: livelock  (trylock and retry)\n");
    fprintf(stderr, "  2: hierarchy (lower-numbered fork first)\n");
    return 1;
  }

  strategy = (int)strtol(argv[1], NULL, 10);
  if (strategy < 0 || strategy > 2) {
    fprintf(stderr, "Error: strategy must be 0, 1 or 2\n");
    return 1;
  }

  const char* names[] = {"deadlock (naive)", "livelock (trylock + retry)",
                         "hierarchy (solution)"};
  printf("Dining Philosophers\n");
  printf("Philosophers: %d, Meals each: %d, Strategy: %s\n\n", NUM_PHILOSOPHERS,
         MEALS_TO_EAT, names[strategy]);
  if (strategy != 2) {
    printf(
        "This strategy is not expected to terminate. Press Ctrl+C to "
        "stop.\n\n");
  }

  for (int i = 0; i < NUM_PHILOSOPHERS; ++i) {
    pthread_mutex_init(&forks[i], NULL);
  }

  pthread_t philosophers[NUM_PHILOSOPHERS];
  for (long i = 0; i < NUM_PHILOSOPHERS; ++i) {
    if (pthread_create(&philosophers[i], NULL, Philosopher, (void*)i) != 0) {
      fprintf(stderr, "Error: pthread_create failed\n");
      return 1;
    }
  }
  for (int i = 0; i < NUM_PHILOSOPHERS; ++i) {
    pthread_join(philosophers[i], NULL);
  }

  for (int i = 0; i < NUM_PHILOSOPHERS; ++i) {
    pthread_mutex_destroy(&forks[i]);
  }

  printf("\nAll philosophers finished.\n");
  return 0;
}

In [ ]:
dining = compile_c(f"{SRC_DIR}/pthread_dining_philosophers.c",
                   f"{SRC_DIR}/dining")

## 7. 策略零：一次真实的死锁

下面运行策略零，限时 3 秒。**程序预期不会自行结束**，届时会被强制终止——这正是要观察的现象。

In [ ]:
hung0, samples0, text0 = monitor(dining, 0, timeout=3.0, interval=1.0)
r0 = report("策略零 · 先左后右，阻塞等待", hung0, samples0, text0)

print("\n程序输出：")
for line in text0.splitlines():
    print("   ", line)

### 结果解读

若程序被强制终止，且采样表中**输出行数在某一时刻之后不再增长、完成进餐次数始终为 0**，即可判定发生了死锁。

请注意最后几行输出的形态：每位哲学家都停留在 `waiting for RIGHT fork`，即**全部持有左叉、全部等待右叉**。等待链首尾相接，构成一个闭合的环。

同时观察 **CPU 时间**：它几乎不再增长。这是死锁的重要特征——线程并非在忙碌，而是全部处于**内核阻塞态**，被挂在互斥量的等待队列上，不消耗任何处理器时间。

### 💡 如何在真实项目中诊断死锁

程序挂起时，可用如下方法定位：

```bash
./dining 0 &                        # 后台运行
gdb -p $(pgrep dining)              # 附加到该进程
(gdb) thread apply all bt           # 打印所有线程的调用栈
```

若看到多个线程都停在 `pthread_mutex_lock` 上，且它们等待的锁互相构成环，即可确认死锁。

也可以直接查看线程状态，无需 `gdb`：

```bash
cat /proc/$(pgrep dining)/task/*/stat | awk '{print $1, $3}'
```

状态字段为 `S`（可中断睡眠）且 CPU 时间不增长，即为阻塞态的典型表现。

> **死锁的判定特征**：线程全部阻塞、CPU 占用趋近于零、输出完全冻结。

## 8. 策略一：从死锁到活锁

策略一用 `trylock` 破坏了「占有并等待」，死锁不会再发生。下面运行并观察其实际行为，限时 3 秒。

In [ ]:
hung1, samples1, text1 = monitor(dining, 1, timeout=3.0, interval=1.0)
r1 = report("策略一 · trylock 失败即重试", hung1, samples1, text1)

# print("\n程序输出：")
# for line in text1.splitlines():
#     print("   ", line)

retries = text1.count("busy, releasing")
print(f"「放下左叉重试」共发生 {retries} 次")

if hung1:
    grew = len({n for _, n, _, _ in samples1}) > 1
    if grew and text1.count("finished a meal") == 0:
        print("\n判定：输出持续增长但无人完成进餐 —— 这是活锁。")
        print(
            "与策略零的区别在于：线程都在运行，程序看起来「很忙」，却没有任何实质进展。"
        )
    else:
        print(
            "\n判定：程序未能在限时内完成，但取得了部分进展，属于进展缓慢而非严格活锁。"
        )
else:
    print(f"\n本次运行程序自行结束，未形成持久活锁（本机核心数 = {os.cpu_count()}）。")
    print("活锁的成立需要多个线程长时间保持同步节拍。核心数较少时，线程被分时轮转，")
    print("对称性容易被打破，因而不易复现。这不代表策略一是安全的：")
    print("它对进展没有任何保证，在多核平台上完全可能长时间停滞。")
    print("请在鲲鹏多核平台上重跑本单元，并尝试加大哲学家人数。")


### 死锁与活锁的对比

<!--
| | 死锁（Deadlock） | 活锁（Livelock） |
|---|---|---|
| 线程状态 | 全部**阻塞**在锁的等待队列上 | 全部**运行**，反复获取与释放 |
| 输出表现 | 完全冻结 | 持续滚动，但无实质进展 |
| 是否取得进展 | 否 | 否 |
| 表面现象 | 程序卡死 | 程序「很忙」，但业务指标不动 |
| 排查难度 | 较低（`gdb` 一看便知） | **较高**（监控上看一切正常） |
-->
<table>
  <thead>
    <tr>
      <th style="text-align: left;"></th>
      <th style="text-align: left;">死锁（Deadlock）</th>
      <th style="text-align: left;">活锁（Livelock）</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;">线程状态</td>
      <td style="text-align: left;">全部<strong>阻塞</strong>在锁的等待队列上</td>
      <td style="text-align: left;">全部<strong>运行</strong>，反复获取与释放</td>
    </tr>
    <tr>
      <td style="text-align: left;">输出表现</td>
      <td style="text-align: left;">完全冻结</td>
      <td style="text-align: left;">持续滚动，但无实质进展</td>
    </tr>
    <tr>
      <td style="text-align: left;">是否取得进展</td>
      <td style="text-align: left;">否</td>
      <td style="text-align: left;">否</td>
    </tr>
    <tr>
      <td style="text-align: left;">表面现象</td>
      <td style="text-align: left;">程序卡死</td>
      <td style="text-align: left;">程序「很忙」，但业务指标不动</td>
    </tr>
    <tr>
      <td style="text-align: left;">排查难度</td>
      <td style="text-align: left;">较低（<code>gdb</code> 一看便知）</td>
      <td style="text-align: left;"><strong>较高</strong>（监控上看一切正常）</td>
    </tr>
  </tbody>
</table>

**活锁比死锁更难发现**：系统监控面板上进程存活、日志仍在滚动，只有业务指标长期不动。

### ⚠️ 关于活锁的 CPU 占用

教材中常把「CPU 占用接近 100%」列为活锁的特征。这一表述需要限定条件：**它取决于重试路径是否包含退避等待**。

- 若重试是**紧凑的自旋循环**（无任何等待），线程持续占用处理器，CPU 占用确实接近 100%；
- 本实验的策略一虽然失败后立即重试，但每一轮循环中都包含一次 sleep_ms(100)（用于保证并发窗口），线程大部分时间处于睡眠状态，因此**CPU 占用同样不高**。

两者都是活锁。因此判定活锁的依据应当是**「线程在运行且状态不断变化，但系统整体不取得进展」**，而非 CPU 占用率。CPU 占用只是一个随实现而变的伴随现象。

上一单元的采样表同时给出了输出行数与 CPU 时间两项指标，正是为了让这一区别可被直接观察。

## 9. 策略二：资源分级

最后运行策略二。**该策略预期能够正常结束**，五位哲学家各完成 3 餐，共 15 次。

In [ ]:
hung2, samples2, text2 = monitor(dining, 2, timeout=30.0, interval=1.0)
r2 = report("策略二 · 资源分级（先取编号较小的叉子）", hung2, samples2, text2)

if not hung2 and text2.count("finished a meal") == 15:
    print("\n✅ 全部哲学家顺利完成进餐，程序正常退出。")
    print("   循环等待条件被破坏，死锁在数学上已被排除，与调度时序无关。")
else:
    print("\n⚠️  未达到预期。请检查是否因系统负载过高导致超时，可加大 timeout 后重试。")


## 10. 三种策略的汇总对比

In [ ]:
import matplotlib.pyplot as plt

names = ["Strategy 0\nleft then right", "Strategy 1\ntrylock + retry",
         "Strategy 2\nhierarchy"]
meals = [
    text0.count("finished a meal"),
    text1.count("finished a meal"),
    text2.count("finished a meal"),
]
hungs = [hung0, hung1, hung2]

print(f"{'策略':<16}{'是否自行结束':>14}{'完成进餐':>10}{'破坏的条件':>14}")
print("-" * 58)
broke = ["无", "占有并等待", "循环等待"]
for nm, h, m, b in zip(
    ["策略零 先左后右", "策略一 trylock 重试", "策略二 资源分级"], hungs, meals, broke
):
    print(f"{nm:<16}{'否（被终止）' if h else '是':>14}{m:>10}{b:>14}")

colors = [
    "#C7000B" if h else ("#2E7D32" if m == 15 else "#E8833A")
    for h, m in zip(hungs, meals)
]
fig, ax = plt.subplots(figsize=(8, 4.2))
bars = ax.bar(names, meals, color=colors)
ax.axhline(15, ls="--", c="gray", lw=1.2,
           label="Expected 15 meals (5 philosophers x 3)")
for b, m, h in zip(bars, meals, hungs):
    tag = f"{m}" + ("\nhung" if h else "")
    ax.text(
        b.get_x() + b.get_width() / 2,
        b.get_height(),
        tag,
        ha="center",
        va="bottom",
        fontsize=9,
    )
ax.set_ylabel("Meals completed")
ax.set_ylim(0, 18)
ax.set_title(f"Progress of the three fork-taking strategies "
             f"({os.cpu_count()} cores)")
ax.grid(axis="y", alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

### 对照总结

<!--
| | 策略零 先左后右 | 策略一 trylock 重试 | 策略二 资源分级 |
|---|---|---|---|
| 破坏的条件 | 无 | 占有并等待 | **循环等待** |
| 结果 | **死锁** | **活锁**（概率性） | **正确** |
| 线程状态 | 全部阻塞 | 全部运行 | 正常推进 |
| 是否保证进展 | 否 | **否** | **是** |
| 额外运行开销 | — | 反复加锁、解锁与重试 | **几乎为零** |
| 代码改动量 | — | 引入重试循环与退避 | **两行比较** |
-->
<table>
  <thead>
    <tr>
      <th style="text-align: left;"></th>
      <th style="text-align: left;">策略零 先左后右</th>
      <th style="text-align: left;">策略一 trylock 重试</th>
      <th style="text-align: left;">策略二 资源分级</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;">破坏的条件</td>
      <td style="text-align: left;">无</td>
      <td style="text-align: left;">占有并等待</td>
      <td style="text-align: left;"><strong>循环等待</strong></td>
    </tr>
    <tr>
      <td style="text-align: left;">结果</td>
      <td style="text-align: left;"><strong>死锁</strong></td>
      <td style="text-align: left;"><strong>活锁</strong>（概率性）</td>
      <td style="text-align: left;"><strong>正确</strong></td>
    </tr>
    <tr>
      <td style="text-align: left;">线程状态</td>
      <td style="text-align: left;">全部阻塞</td>
      <td style="text-align: left;">全部运行</td>
      <td style="text-align: left;">正常推进</td>
    </tr>
    <tr>
      <td style="text-align: left;">是否保证进展</td>
      <td style="text-align: left;">否</td>
      <td style="text-align: left;"><strong>否</strong></td>
      <td style="text-align: left;"><strong>是</strong></td>
    </tr>
    <tr>
      <td style="text-align: left;">额外运行开销</td>
      <td style="text-align: left;">—</td>
      <td style="text-align: left;">反复加锁、解锁与重试</td>
      <td style="text-align: left;"><strong>几乎为零</strong></td>
    </tr>
    <tr>
      <td style="text-align: left;">代码改动量</td>
      <td style="text-align: left;">—</td>
      <td style="text-align: left;">引入重试循环与退避</td>
      <td style="text-align: left;"><strong>两行比较</strong></td>
    </tr>
  </tbody>
</table>

策略二相较策略零只多了两行比较语句，**没有任何运行时开销**，却彻底排除了死锁。

> 这是本实验最值得记住的一点：
> **正确的并发设计往往并不比错误的更复杂、更慢。**
> 死锁不是「性能与正确性之间的权衡」，它纯粹是设计缺陷。

## 11. 结果分析

本实验建立了三项认识：

**① 每一处同步都正确，不等于程序整体正确。** 策略零中每一次 `pthread_mutex_lock` 的用法都无可指摘，数据竞争也确实被消除了。问题出在**多个正确的局部行为组合成了一个错误的全局行为**。并发程序的正确性必须在**系统层面**论证，而不能仅凭逐处检查。

**② 「不会死锁」与「一定能完成」是两个不同的性质。** 策略一确实消除了死锁，却无法保证任何线程最终取得进展。在形式化方法中，这两者分别对应**安全性**（safety：坏事永不发生）与**活性**（liveness：好事终将发生）。破坏死锁的四条件之一只保证了前者。

**③ 破坏「循环等待」是唯一有保证的路径。** 资源分级不依赖概率、不依赖调度、不依赖退避策略的调参，它把死锁的可能性在**数学上**排除了。

### 🎓 结论

三种策略中，只有资源分级给出了**确定性**的保证。这提示我们，处理并发故障时应当优先寻找**结构性**的解法，而非依赖概率性的缓解手段：

<!--
| 手段 | 性质 | 可靠性 |
|---|---|---|
| 引入随机退避、拉长重试间隔 | 概率性缓解 | 降低发生率，无保证 |
| 增加超时与重试 | 概率性缓解 | 掩盖问题，无保证 |
| **为所有锁定义全局获取顺序** | **结构性排除** | **有保证** |
-->
<table>
  <thead>
    <tr>
      <th style="text-align: left;">手段</th>
      <th style="text-align: left;">性质</th>
      <th style="text-align: left;">可靠性</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;">引入随机退避、拉长重试间隔</td>
      <td style="text-align: left;">概率性缓解</td>
      <td style="text-align: left;">降低发生率，无保证</td>
    </tr>
    <tr>
      <td style="text-align: left;">增加超时与重试</td>
      <td style="text-align: left;">概率性缓解</td>
      <td style="text-align: left;">掩盖问题，无保证</td>
    </tr>
    <tr>
      <td style="text-align: left;"><strong>为所有锁定义全局获取顺序</strong></td>
      <td style="text-align: left;"><strong>结构性排除</strong></td>
      <td style="text-align: left;"><strong>有保证</strong></td>
    </tr>
  </tbody>
</table>

在实际工程中，前两类手段常被用作临时措施，但它们只是把故障出现的周期拉长，并未消除故障。真正的修复必须归结为第三类。

## 12. 🔧 动手练习

请修改代码、重新编译并运行，观察行为的变化：

1. 删去策略零中的 `sleep_ms(100)`，重复运行 10 次，统计死锁发生的比例，解释该延迟为何会提高死锁的复现率。
2. 让策略零死锁后，用 `gdb -p $(pgrep dining)` 附加进程并执行 `thread apply all bt`，记录各线程停在哪一行，据此画出等待关系图并指出其中的环。
3. 在策略一的重试路径中加入随机退避（如失败后 `sleep_ms(rand() % 100)`），多次运行并统计活锁的发生率变化，说明随机退避为何只能缓解而不能根治。
4. 实现第四种策略：用一个初值为 4 的信号量限制同时入座的哲学家人数（五人中最多四人同时取叉）。说明它破坏了四条件中的哪一个，并验证其正确性。
5. 把哲学家人数依次改为 2、3、10、50，测量策略二的总耗时，分析并发度随人数的变化趋势。
6. 删去策略一中的 `sleep_ms(100)`，重复运行 10 次，记录活锁是否仍然复现。结合策略零中的同一行延迟，说明「并发窗口」的大小如何决定故障的可复现性。

## 13. 🤔 思考题

- 策略一破坏了「占有并等待」，因此不会死锁，但引入了活锁。请说明：为什么「不会死锁」不等于「一定能完成」？这两者在形式化方法中分别对应什么性质？
- 策略二中，哲学家 $P_4$ 需要先取叉子 0 再取叉子 4，而其余人先取左叉。这种不对称是否会导致某位哲学家长期吃不到饭（**饥饿**，starvation）？请分析并说明饥饿与死锁的区别。
- 资源分级要求「全局一致的编号」。在一个大型系统中，锁可能分散于不同模块、由不同团队维护。请提出一个可落地的工程方案来保证编号的全局一致性。
- 若两把锁的地址在运行时才确定（例如转账时锁住两个账户对象），如何应用资源分级？直接按指针地址大小排序是否可行？有什么陷阱？
- 本实验的三种策略中只有策略二保证了进展，但它要求**预先知道**需要的全部锁。若某个操作在执行过程中才发现还需要第三把锁，且其编号更小，应当如何处理？请给出一种方案。

## 14. 小结与后续

本实验通过同一问题的三种取叉策略，完整呈现了多锁场景下的两类故障及其正确解法：

<!--
| 策略 | 破坏的条件 | 结果 | 新增知识点 |
|---|---|---|---|
| **策略零** | 无 | 死锁 | Coffman 四条件、死锁的诊断方法 |
| **策略一** | 占有并等待 | 活锁 | `pthread_mutex_trylock`、安全性与活性之别、随机退避 |
| **策略二** | **循环等待** | **正确** | 资源分级、全局加锁顺序规范 |
-->
<table>
  <thead>
    <tr>
      <th style="text-align: left;">策略</th>
      <th style="text-align: left;">破坏的条件</th>
      <th style="text-align: left;">结果</th>
      <th style="text-align: left;">新增知识点</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;"><strong>策略零</strong></td>
      <td style="text-align: left;">无</td>
      <td style="text-align: left;">死锁</td>
      <td style="text-align: left;">Coffman 四条件、死锁的诊断方法</td>
    </tr>
    <tr>
      <td style="text-align: left;"><strong>策略一</strong></td>
      <td style="text-align: left;">占有并等待</td>
      <td style="text-align: left;">活锁</td>
      <td style="text-align: left;"><code>pthread_mutex_trylock</code>、安全性与活性之别、随机退避</td>
    </tr>
    <tr>
      <td style="text-align: left;"><strong>策略二</strong></td>
      <td style="text-align: left;"><strong>循环等待</strong></td>
      <td style="text-align: left;"><strong>正确</strong></td>
      <td style="text-align: left;">资源分级、全局加锁顺序规范</td>
    </tr>
  </tbody>
</table>

至此，互斥量这一工具的能力与边界都已完整呈现：

- **实验三**说明了它**能**解决什么——用最小的临界区消除数据竞争；
- **本实验**说明了它**用不好会**造成什么——多锁场景下的死锁与活锁，以及如何从结构上避免。

➡️ **后续内容：实验五 消息传递：四种同步策略的递进**。到目前为止，我们处理的都是「**谁能进入临界区**」这一类问题，互斥量是称手的工具。实验五将提出一类**互斥量无法表达**的需求：线程需要**等待某个事件发生**。届时会看到，即使每个线程写的是互不重叠的位置（根本不存在数据竞争），程序依然可能出错——因为缺少的不是互斥，而是**顺序保证**。互斥量对此无能为力，需要引入新的同步原语。